# COSC-4117EL A3 — Face Liveness Detection
## Experiments: Architecture Comparison

**Group 8**

This notebook compares several model architectures on the same face-liveness task:

| # | Experiment | Models |
|---|---|---|
| 1 | Deeper ResNets | ResNet-34, ResNet-50 |
| 2 | YOLO classifiers | yolo11n-cls, yolo26n-cls |

All models are evaluated on the same held-out test split (same seed as the training notebook).
A final comparison table and charts are produced at the end.

**Prerequisites**: Run `COSC_4117EL_A3_G8-train.ipynb` first so that `data/processed/` and
`models/resnet18_best.pth` / `models/resnet18_test_results.json` already exist.

> *Code assisted with Claude Code (Anthropic) — cited per course AI-use policy.*

In [ ]:
# Uncomment to install dependencies
# %pip install torch torchvision ultralytics mediapipe opencv-python pillow scikit-learn matplotlib seaborn tqdm pandas --quiet

In [ ]:
import os
import json
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models
import torchvision.transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

print(f'PyTorch: {torch.__version__}')

## Shared Setup

In [ ]:
CFG = {
    'processed_dir': 'data/processed',
    'models_dir':    'models',
    'yolo_dir':      'data/yolo_cls',
    'image_size':    224,
    'batch_size':    32,
    'epochs':        20,
    'lr':            1e-4,
    'weight_decay':  1e-4,
    'patience':      5,
    'val_split':     0.15,
    'test_split':    0.15,
    'seed':          42,
}

CLASS_NAMES   = {0: 'spoof', 1: 'live'}
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])

if torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

os.makedirs(CFG['models_dir'], exist_ok=True)
print(f'Device: {DEVICE}')

In [ ]:
# ── Dataset (mirrors training notebook; identical seed → identical split) ────
class LivenessDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths, self.labels, self.transform = list(paths), list(labels), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])


eval_tf = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size'])),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])
train_tf = T.Compose([
    T.Resize((CFG['image_size'], CFG['image_size'])),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    T.RandomGrayscale(p=0.02),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

all_paths, all_labels = [], []
for label, cls in [(1, 'live'), (0, 'spoof')]:
    for p in sorted((Path(CFG['processed_dir']) / cls).glob('*.png')):
        all_paths.append(str(p))
        all_labels.append(label)

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)

val_test = CFG['val_split'] + CFG['test_split']
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    all_paths, all_labels,
    test_size=val_test, stratify=all_labels, random_state=CFG['seed']
)
val_frac = CFG['val_split'] / val_test
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp,
    test_size=1 - val_frac, stratify=y_tmp, random_state=CFG['seed']
)

train_ds = LivenessDataset(X_tr,  y_tr,  transform=train_tf)
val_ds   = LivenessDataset(X_val, y_val, transform=eval_tf)
test_ds  = LivenessDataset(X_te,  y_te,  transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

print(f'train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}')

## Shared Helpers

In [ ]:
def build_resnet(arch: str, num_classes: int = 2, frozen: bool = True) -> nn.Module:
    weights_map  = {
        'resnet18': tv_models.ResNet18_Weights.IMAGENET1K_V1,
        'resnet34': tv_models.ResNet34_Weights.IMAGENET1K_V1,
        'resnet50': tv_models.ResNet50_Weights.IMAGENET1K_V2,
    }
    factory_map  = {
        'resnet18': tv_models.resnet18,
        'resnet34': tv_models.resnet34,
        'resnet50': tv_models.resnet50,
    }
    m = factory_map[arch](weights=weights_map[arch])
    if frozen:
        for p in m.parameters():
            p.requires_grad = False
    in_f = m.fc.in_features
    m.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(in_f, num_classes))
    return m.to(DEVICE)


def train_resnet(model, train_loader, val_loader, save_path, cfg):
    """Train only the FC head; backbone remains frozen throughout."""
    criterion = nn.CrossEntropyLoss()

    # Only head parameters are passed to the optimizer
    optimizer = optim.Adam(
        model.fc.parameters(),
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=cfg['epochs'], eta_min=1e-6
    )
    best_val, patience_cnt = float('inf'), 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        run_loss = 0.0
        for imgs, labels in tqdm(train_loader, desc=f'Ep {epoch:02d} train', leave=False):
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.fc.parameters(), 1.0)
            optimizer.step()
            run_loss += loss.item() * imgs.size(0)
        tr_loss = run_loss / len(train_loader.dataset)

        model.eval()
        v_loss = v_correct = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                out = model(imgs)
                v_loss   += criterion(out, labels).item() * imgs.size(0)
                v_correct += (out.argmax(1) == labels).sum().item()
        v_loss /= len(val_loader.dataset)
        v_acc   = v_correct / len(val_loader.dataset)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)
        scheduler.step()
        print(f'Ep {epoch:02d}  tr={tr_loss:.4f}  val={v_loss:.4f}  acc={v_acc:.4f}')

        if v_loss < best_val:
            best_val, patience_cnt = v_loss, 0
            torch.save({'epoch': epoch, 'state_dict': model.state_dict(),
                        'val_loss': best_val, 'val_acc': v_acc,
                        'class_names': CLASS_NAMES, 'image_size': cfg['image_size']},
                       save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= cfg['patience']:
                print(f'Early stop @ ep {epoch}')
                break
    return history


def eval_resnet(model, loader):
    model.eval()
    preds, trues = [], []
    t0 = time.perf_counter()
    with torch.no_grad():
        for imgs, labels in loader:
            preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().tolist())
            trues.extend(labels.tolist())
    fps = len(preds) / (time.perf_counter() - t0)
    preds, trues = np.array(preds), np.array(trues)
    acc              = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(trues, preds, average='macro')
    return {'accuracy': float(acc), 'precision': float(prec),
            'recall': float(rec), 'f1': float(f1), 'fps': float(fps)}


def measure_fps_single(model, image_size=224, n_runs=200):
    model.eval()
    dummy = torch.randn(1, 3, image_size, image_size).to(DEVICE)
    with torch.no_grad():
        for _ in range(20):  # warm-up
            _ = model(dummy)
    t0 = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy)
    return n_runs / (time.perf_counter() - t0)

## Baseline: Load ResNet-18 Results

In [ ]:
results = []

r18_json = os.path.join(CFG['models_dir'], 'resnet18_test_results.json')
r18_ckpt = os.path.join(CFG['models_dir'], 'resnet18_best.pth')

if os.path.exists(r18_json):
    with open(r18_json) as f:
        r18 = json.load(f)
    if os.path.exists(r18_ckpt):
        m18 = build_resnet('resnet18', frozen=False)
        ck  = torch.load(r18_ckpt, map_location=DEVICE, weights_only=False)
        m18.load_state_dict(ck['state_dict'])
        r18['fps'] = measure_fps_single(m18)
    else:
        r18['fps'] = float('nan')
    results.append(r18)
    print(f"ResNet-18 baseline: acc={r18['accuracy']:.4f}  f1={r18['f1']:.4f}  fps={r18.get('fps', 0):.1f}")
else:
    print('WARNING: run the training notebook first to generate resnet18_test_results.json')

## Experiment 1a — ResNet-34

In [ ]:
print('=== ResNet-34 ===')
m34     = build_resnet('resnet34', frozen=True)
path_34 = os.path.join(CFG['models_dir'], 'resnet34_best.pth')

if os.path.exists(path_34):
    print('Checkpoint found — loading.')
    ck = torch.load(path_34, map_location=DEVICE, weights_only=False)
    m34.load_state_dict(ck['state_dict'])
else:
    train_resnet(m34, train_loader, val_loader, path_34, CFG)
    ck = torch.load(path_34, map_location=DEVICE, weights_only=False)
    m34.load_state_dict(ck['state_dict'])

r34 = eval_resnet(m34, test_loader)
r34.update({'model': 'ResNet-34', 'fps': measure_fps_single(m34)})
results.append(r34)
print(f"ResNet-34: acc={r34['accuracy']:.4f}  f1={r34['f1']:.4f}  fps={r34['fps']:.1f}")

## Experiment 1b — ResNet-50

In [ ]:
print('=== ResNet-50 ===')
cfg50 = {**CFG, 'batch_size': 16, 'lr': 5e-5}

tl50 = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=0)
vl50 = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=0)

m50     = build_resnet('resnet50', frozen=True)
path_50 = os.path.join(CFG['models_dir'], 'resnet50_best.pth')

if os.path.exists(path_50):
    print('Checkpoint found — loading.')
    ck = torch.load(path_50, map_location=DEVICE, weights_only=False)
    m50.load_state_dict(ck['state_dict'])
else:
    train_resnet(m50, tl50, vl50, path_50, cfg50)
    ck = torch.load(path_50, map_location=DEVICE, weights_only=False)
    m50.load_state_dict(ck['state_dict'])

tel50 = DataLoader(test_ds, batch_size=16, shuffle=False, num_workers=0)
r50 = eval_resnet(m50, tel50)
r50.update({'model': 'ResNet-50', 'fps': measure_fps_single(m50)})
results.append(r50)
print(f"ResNet-50: acc={r50['accuracy']:.4f}  f1={r50['f1']:.4f}  fps={r50['fps']:.1f}")

## Experiment 2 — YOLO Classification Models

Ultralytics YOLO expects a directory tree with one folder per class.
We copy images from our processed split into `data/yolo_cls/{train,val,test}/{live,spoof}/`.

In [ ]:
def build_yolo_split(yolo_dir, splits_dict, class_names):
    """
    Copy processed images into Ultralytics classification folder structure.
    splits_dict: {'train': (paths, labels), 'val': ..., 'test': ...}
    Idempotent — skips existing files.
    """
    for split, (paths, labels) in splits_dict.items():
        for cls_name in class_names.values():
            (Path(yolo_dir) / split / cls_name).mkdir(parents=True, exist_ok=True)

        for src, lbl in tqdm(zip(paths, labels), total=len(paths), desc=split, leave=False):
            dst = Path(yolo_dir) / split / class_names[int(lbl)] / Path(src).name
            if not dst.exists():
                shutil.copy2(src, dst)

    for split in splits_dict:
        for cls_name in class_names.values():
            n = len(list((Path(yolo_dir) / split / cls_name).glob('*.png')))
            print(f'  {split}/{cls_name}: {n}')


build_yolo_split(
    CFG['yolo_dir'],
    {'train': (X_tr, y_tr), 'val': (X_val, y_val), 'test': (X_te, y_te)},
    CLASS_NAMES
)
print('YOLO data ready at', CFG['yolo_dir'])

In [ ]:
def eval_yolo_on_test(weights_path, test_paths, test_labels):
    """
    Run YOLO classification inference on test images and compute metrics.
    Handles class-name order differences (YOLO uses alphabetical folder order).
    """
    from ultralytics import YOLO
    yolo = YOLO(weights_path)

    # YOLO assigns classes alphabetically: live=0, spoof=1
    # Our label convention: live=1, spoof=0  → invert
    yolo_to_ours = {0: 1, 1: 0}

    # Check actual model names to be safe
    if hasattr(yolo.model, 'names'):
        yolo_names = yolo.model.names
        yolo_to_ours = {i: {'live': 1, 'spoof': 0}.get(n, i)
                        for i, n in yolo_names.items()}

    preds = []
    t0    = time.perf_counter()
    for p in tqdm(test_paths, desc='YOLO test', leave=False):
        r    = yolo.predict(p, verbose=False)[0]
        preds.append(yolo_to_ours.get(int(r.probs.top1), int(r.probs.top1)))
    fps = len(preds) / (time.perf_counter() - t0)

    preds = np.array(preds)
    trues = np.array(test_labels, dtype=int)
    acc              = accuracy_score(trues, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(trues, preds, average='macro')
    return {'accuracy': float(acc), 'precision': float(prec),
            'recall': float(rec), 'f1': float(f1), 'fps': float(fps)}

### Experiment 2a — yolo11n-cls

In [ ]:
from ultralytics import YOLO

print('=== yolo11n-cls ===')

y11n_dir  = os.path.join(CFG['models_dir'], 'yolo11n_cls')
y11n_best = os.path.join(y11n_dir, 'weights', 'best.pt')

if not os.path.exists(y11n_best):
    m11n = YOLO('yolo11n-cls.pt')
    m11n.train(
        data=os.path.abspath(CFG['yolo_dir']),
        epochs=CFG['epochs'],
        imgsz=CFG['image_size'],
        batch=CFG['batch_size'],
        device='cpu' if str(DEVICE) == 'mps' else str(DEVICE),
        project=CFG['models_dir'],
        name='yolo11n_cls',
        exist_ok=True,
        verbose=False,
    )
else:
    print('Checkpoint found — skipping training.')

r11n = eval_yolo_on_test(y11n_best, X_te, y_te)
r11n['model'] = 'yolo11n-cls'
results.append(r11n)
print(f"yolo11n-cls: acc={r11n['accuracy']:.4f}  f1={r11n['f1']:.4f}  fps={r11n['fps']:.1f}")

### Experiment 2b — yolo26n-cls

In [ ]:
print('=== yolo26n-cls ===')

y26n_dir  = os.path.join(CFG['models_dir'], 'yolo26n_cls')
y26n_best = os.path.join(y26n_dir, 'weights', 'best.pt')

if not os.path.exists(y26n_best):
    m26n = YOLO('yolo26n-cls.pt')
    m26n.train(
        data=os.path.abspath(CFG['yolo_dir']),
        epochs=CFG['epochs'],
        imgsz=CFG['image_size'],
        batch=CFG['batch_size'],
        device='cpu' if str(DEVICE) == 'mps' else str(DEVICE),
        project=CFG['models_dir'],
        name='yolo26n_cls',
        exist_ok=True,
        verbose=False,
    )
else:
    print('Checkpoint found — skipping training.')

r26n = eval_yolo_on_test(y26n_best, X_te, y_te)
r26n['model'] = 'yolo26n-cls'
results.append(r26n)
print(f"yolo26n-cls: acc={r26n['accuracy']:.4f}  f1={r26n['f1']:.4f}  fps={r26n['fps']:.1f}")

## Comparison Table

In [ ]:
df = pd.DataFrame(results)
cols = ['model', 'accuracy', 'precision', 'recall', 'f1', 'fps']
df = df[[c for c in cols if c in df.columns]].sort_values('f1', ascending=False).reset_index(drop=True)

df_display = df.copy()
for col in ('accuracy', 'precision', 'recall', 'f1'):
    if col in df_display.columns:
        df_display[col] = df_display[col].map('{:.4f}'.format)
if 'fps' in df_display.columns:
    df_display['fps'] = df_display['fps'].map('{:.1f}'.format)

print('=== Model Comparison (sorted by F1) ===')
print(df_display.to_string(index=False))

csv_path = os.path.join(CFG['models_dir'], 'comparison_results.csv')
df.to_csv(csv_path, index=False)
print(f'\nSaved → {csv_path}')

## Visualizations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Grouped bar: Accuracy vs F1
ax = axes[0]
x  = np.arange(len(df))
w  = 0.35
b1 = ax.bar(x - w/2, df['accuracy'].astype(float), w, label='Accuracy',   color='steelblue')
b2 = ax.bar(x + w/2, df['f1'].astype(float),       w, label='F1 (macro)', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(df['model'], rotation=20, ha='right', fontsize=9)
ax.set_ylim(0, 1.08)
ax.set_ylabel('Score')
ax.set_title('Accuracy & F1 by Model')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for bar in list(b1) + list(b2):
    h = bar.get_height()
    ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7)

# Scatter: F1 vs FPS
ax2 = axes[1]
for _, row in df.iterrows():
    ax2.scatter(float(row['fps']), float(row['f1']), s=120, zorder=5)
    ax2.annotate(row['model'], (float(row['fps']), float(row['f1'])),
                 textcoords='offset points', xytext=(5, 3), fontsize=8)
ax2.set_xlabel('Throughput (FPS — single-image inference)')
ax2.set_ylabel('F1 Score (macro)')
ax2.set_title('Accuracy-Speed Trade-off')
ax2.grid(True, alpha=0.3)

plt.suptitle('Face Liveness Detection — Model Comparison', fontsize=13)
plt.tight_layout()
plt.savefig(
    os.path.join(CFG['models_dir'], 'comparison_chart.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
# Confusion matrices for the three ResNet variants
resnet_cfgs = [
    ('ResNet-18', os.path.join(CFG['models_dir'], 'resnet18_best.pth'), 'resnet18'),
    ('ResNet-34', os.path.join(CFG['models_dir'], 'resnet34_best.pth'), 'resnet34'),
    ('ResNet-50', os.path.join(CFG['models_dir'], 'resnet50_best.pth'), 'resnet50'),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, ckpt_path, arch) in zip(axes, resnet_cfgs):
    if not os.path.exists(ckpt_path):
        ax.set_title(f'{name}\n(missing)')
        ax.axis('off')
        continue
    m = build_resnet(arch, frozen=False)
    ck = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    m.load_state_dict(ck['state_dict'])
    m.eval()
    all_p, all_t = [], []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            all_p.extend(m(imgs.to(DEVICE)).argmax(1).cpu().tolist())
            all_t.extend(lbls.tolist())
    cm = confusion_matrix(all_t, all_p)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['spoof', 'live'], yticklabels=['spoof', 'live'], ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')

plt.suptitle('ResNet Confusion Matrices — Test Set', fontsize=12)
plt.tight_layout()
plt.savefig(
    os.path.join(CFG['models_dir'], 'resnet_confusion_matrices.png'),
    dpi=150, bbox_inches='tight'
)
plt.show()